In [1]:
# Célula 1 — Imports
import geopandas as gpd
import pandas as pd
from pathlib import Path

In [2]:
# Célula 2 — Carregar as três camadas
pasta = Path(r"C:\Users\franc\OneDrive\Francisco\Profissional\MBA_Data_Science_ e_Analytics\00_TCC\06_Dados_base\GEO\2024_02_basegeo")

lf = gpd.read_file(pasta / "lf.shp")
bf = gpd.read_file(pasta / "bf.shp")
gl = gpd.read_file(pasta / "gl.shp")

print(f"LF carregado: {len(lf):,} registros")
print(f"BF carregado: {len(bf):,} registros")
print(f"GL carregado: {len(gl):,} registros")
print(f"Total bruto:  {len(lf)+len(bf)+len(gl):,} registros")

LF carregado: 42,016 registros
BF carregado: 155,705 registros
GL carregado: 468 registros
Total bruto:  198,189 registros


In [3]:
# Célula 3 — Padronizar colunas
lf = lf.rename(columns={"IDSMFLF": "ID_ORIG"})
bf = bf.rename(columns={"IDBFLF_": "ID_ORIG"})
gl["ID_ORIG"] = range(len(gl))

lf["CAMADA_ORIG"] = "LF"
bf["CAMADA_ORIG"] = "BF"
gl["CAMADA_ORIG"] = "GL"

cols = ["NUMBLOCO", "SETOR", "QUARTEIRAO", "AREA", "ID_ORIG", "CAMADA_ORIG", "geometry"]
lf = lf[cols]
bf = bf[cols]
gl = gl[cols]

In [4]:
# Célula 4 — União simples (Etapa 1)
lotes_raw = pd.concat([lf, bf, gl], ignore_index=True)
lotes_raw = gpd.GeoDataFrame(lotes_raw, geometry="geometry", crs=lf.crs)
print(f"\nEtapa 1 — União simples: {len(lotes_raw):,} registros")


Etapa 1 — União simples: 198,189 registros


In [5]:
# Célula 5 — Aplicar regras de limpeza (Etapa 2)
# Regra ①: geometry nula
mask_r1 = lotes_raw.geometry.isna()

# Regra ②: AREA = 0
mask_r2 = lotes_raw["AREA"] == 0

# Regra ③: registro fantasma (SETOR=0 E QUARTEIRAO=0)
mask_r3 = (lotes_raw["SETOR"] == 0) & (lotes_raw["QUARTEIRAO"] == 0)

# Regra ④: NUMBLOCO duplicado com mesma AREA (exceto 000000000000)
mask_r4 = (
    lotes_raw.duplicated(subset=["NUMBLOCO", "AREA"], keep="first") &
    lotes_raw["NUMBLOCO"].ne("000000000000")
)

print(f"\nEtapa 2 — Registros removidos por regra:")
print(f"  ① Geometry nula:      {mask_r1.sum():>6,}")
print(f"  ② AREA = 0:           {mask_r2.sum():>6,}")
print(f"  ③ Registro fantasma:  {mask_r3.sum():>6,}")
print(f"  ④ NUMBLOCO duplicado: {mask_r4.sum():>6,}")

# Verificar sobreposição entre regras
mask_remover = mask_r1 | mask_r2 | mask_r3 | mask_r4
print(f"\n  Total único a remover: {mask_remover.sum():>6,}")
print(f"  Total a manter:        {(~mask_remover).sum():>6,}")


Etapa 2 — Registros removidos por regra:
  ① Geometry nula:           4
  ② AREA = 0:               45
  ③ Registro fantasma:      12
  ④ NUMBLOCO duplicado:      9

  Total único a remover:     63
  Total a manter:        198,126


In [6]:
# Célula 6 — Aplicar limpeza e campos derivados (Etapas 3 e 4)
lotes = lotes_raw[~mask_remover].copy().reset_index(drop=True)

lotes["LOTE_PAI"]     = lotes["NUMBLOCO"].str[:8]
lotes["SUBLOTE"]      = lotes["NUMBLOCO"].str[8:]
lotes["TEM_CADASTRO"] = lotes["NUMBLOCO"] != "000000000000"
lotes["AREA_GEO"]     = lotes.geometry.area.round(2)

In [7]:
# Célula 7 — Resumo final
print("\n" + "="*55)
print("CAMADA LOTES — CONSOLIDADA")
print("="*55)
print(f"Total de registros:      {len(lotes):,}")
print(f"Com cadastro:            {lotes['TEM_CADASTRO'].sum():,}")
print(f"Sem cadastro (000...):   {(~lotes['TEM_CADASTRO']).sum():,}")
print(f"\nPor camada de origem:")
print(lotes["CAMADA_ORIG"].value_counts().to_string())
print(f"\nSublotes (SUBLOTE ≠ 0000): {(lotes['SUBLOTE'] != '0000').sum():,}")
print(f"\nCRS: {lotes.crs}")
print(f"\nAmostra (3 registros):")
print(lotes[["NUMBLOCO","LOTE_PAI","SUBLOTE","TEM_CADASTRO",
             "AREA","AREA_GEO","SETOR","QUARTEIRAO","CAMADA_ORIG"]].head(3).to_string())


CAMADA LOTES — CONSOLIDADA
Total de registros:      198,126
Com cadastro:            172,153
Sem cadastro (000...):   25,973

Por camada de origem:
CAMADA_ORIG
BF    155693
LF     42010
GL       423

Sublotes (SUBLOTE ≠ 0000): 7,086

CRS: PROJCS["TM-POA",GEOGCS["SIRGAS 2000",DATUM["Sistema_de_Referencia_Geocentrico_para_las_AmericaS_2000",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6674"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",-51],PARAMETER["scale_factor",0.999995],PARAMETER["false_easting",300000],PARAMETER["false_northing",5000000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]

Amostra (3 registros):


       NUMBLOCO  LOTE_PAI SUBLOTE  TEM_CADASTRO          AREA  AREA_GEO  SETOR  QUARTEIRAO CAMADA_ORIG
0  001636850000  00163685    0000          True  38232.252691  38233.13     37          62          LF
1  007025580000  00702558    0000          True   8035.048352   8033.98     37          62          LF
2  007027320000  00702732    0000          True    351.971906    352.16     49         199          LF


In [8]:
# Célula 8 — Salvar
saida = pasta / "lotes_consolidados.gpkg"
lotes.to_file(saida, driver="GPKG")
print(f"\n✅ Arquivo salvo: {saida}")
print(f"   Tamanho: {saida.stat().st_size / 1024 / 1024:.1f} MB")


✅ Arquivo salvo: C:\Users\franc\OneDrive\Francisco\Profissional\MBA_Data_Science_ e_Analytics\00_TCC\06_Dados_base\GEO\2024_02_basegeo\lotes_consolidados.gpkg
   Tamanho: 83.1 MB
